## Baseline Readmission Rate
After examining the Excel output (produced from exploration.ipynb), each category is always around 13% for the readmission rates regardless of the value.
So a model predicting binary value (readmitted within or more than 30 days) would achieve 87% accuracy as well.
Therefore F1 score and recall will be the performance metrics instead of accuracy.

## Readmission Rate Distributions from readmission_by_col.xlsx
Number of inpatient visits might be the strongest indicator since as this number increases, the more likely they will be readmitted because it is indictor of severity or the chronic level of diabetes.
Contrary to this, time spent in hospital does not visibly guard against readmission since the rate stays relatively stable as time spent increases.

In [1]:
import pandas as pd, numpy as np
df = pd.DataFrame()
df = pd.read_csv(r'data/diabetic_data.csv')
df.shape

(101766, 50)

In [2]:
df = df.replace('?', np.nan)
df.isnull().sum().sort_values(ascending=False)

weight                      98569
max_glu_serum               96420
A1Cresult                   84748
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
encounter_id                    0
troglitazone                    0
tolbutamide                     0
pioglitazone                    0
rosiglitazone                   0
acarbose                        0
miglitol                        0
citoglipton                     0
tolazamide                      0
examide                         0
glipizide                       0
insulin                         0
glyburide-metformin             0
glipizide-metformin             0
glimepiride-pioglitazone        0
metformin-rosiglitazone         0
metformin-pioglitazone          0
change                          0
diabetesMed                     0
glyburide                       0
repaglinide   

In [3]:
df = df.drop(['weight', 'max_glu_serum', 'medical_specialty'], axis=1)
# after inspecting the Excel notebook, these categories had too many unknowns so were dropped
cols_to_fill = ['payer_code', 'race', 'diag_1', 'diag_2', 'diag_3']
# filling missing values before EDA
df[cols_to_fill] = df[cols_to_fill].replace(np.nan, 'Unknown')

In [4]:
df.apply(lambda x: x.value_counts(normalize=True).max()).sort_values()
# for each feature, what is the percentage of the most common value?
# so with this, we can see which cols have almost no variation in values 

encounter_id                0.000010
patient_nbr                 0.000393
num_lab_procedures          0.031523
num_medications             0.059804
diag_2                      0.066348
diag_1                      0.067429
diag_3                      0.113545
time_in_hospital            0.174479
age                         0.256156
payer_code                  0.395574
num_procedures              0.458424
insulin                     0.465607
A1Cresult                   0.482783
number_diagnoses            0.486155
admission_type_id           0.530531
gender                      0.537586
change                      0.538048
readmitted                  0.539119
admission_source_id         0.564963
discharge_disposition_id    0.591887
number_inpatient            0.664564
race                        0.747784
diabetesMed                 0.770031
metformin                   0.803589
number_outpatient           0.835515
glipizide                   0.875341
number_emergency            0.888145
g

In [5]:
# cols that a value that accounts for more than 90% majority are dropped 
for col in df.columns:
    if df[col].value_counts(normalize=True).max() > 0.90:
        print(f"{col}")
        df = df.drop(col, axis=1)

repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
examide
citoglipton
glyburide-metformin
glipizide-metformin
glimepiride-pioglitazone
metformin-rosiglitazone
metformin-pioglitazone


In [6]:
# count number of values with decimal points before binning codes 
print(df['diag_1'].str.contains(r'\.').sum())
print(df['diag_2'].str.contains(r'\.').sum())
print(df['diag_3'].str.contains(r'\.').sum())

8522
6723
5603


In [7]:
# bin codes to go from dtype:object to dtype:str
def bin_codes(code):
    if code.startswith('V'):
        return 'V_codes'
    elif code.startswith('E'):
        return 'E_codes'
    elif code.split('.')[0].isnumeric():
        num = int(code.split('.')[0])
        ranges = [
            (1, 139, 'Infectious and parasitic diseases'),
            (140, 239, 'Neoplasms'),
            (240, 279, 'Endocrine, nutritional and metabolic diseases, and immunity disorders'),
            (280, 289, 'Diseases of the blood and blood-forming organs'),
            (290, 319, 'Mental disorders'),
            (320, 389, 'Diseases of the nervous system and sense organs'),
            (390, 459, 'Diseases of the circulatory system'),
            (460, 519, 'Diseases of the respiratory system'),
            (520, 579, 'Diseases of the digestive system'),
            (580, 629, 'Diseases of the genitourinary system'),
            (630, 679, 'Complications of pregnancy, childbirth, and the puerperium'),
            (680, 709, 'Diseases of the skin and subcutaneous tissue'),
            (710, 739, 'Diseases of the musculoskeletal system and connective tissue'),
            (740, 759, 'Congenital anomalies'),
            (760, 779, 'Certain conditions originating in the perinatal period'),
            (780, 799, 'Symptoms, signs, and ill-defined conditions'),
            (800, 999, 'Injury and poisoning')]
        
        for low, high, label in ranges:
            if low <= num <= high:
                return label
    else:
        return 'Unknown'

In [8]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(bin_codes)

df['diag_3'].value_counts()                 # check that all 101766 values were binned 

diag_3
Diseases of the circulatory system                                       29918
Endocrine, nutritional and metabolic diseases, and immunity disorders    26308
Diseases of the respiratory system                                        6774
Diseases of the genitourinary system                                      6327
Symptoms, signs, and ill-defined conditions                               4523
V_codes                                                                   3814
Diseases of the digestive system                                          3572
Mental disorders                                                          3136
Diseases of the blood and blood-forming organs                            2490
Diseases of the skin and subcutaneous tissue                              2488
Injury and poisoning                                                      1946
Diseases of the musculoskeletal system and connective tissue              1915
Infectious and parasitic diseases            

In [9]:
df['readmitted'] = df['readmitted'].replace({ 'NO': 0, '>30': 0, '<30': 1}).astype(int)
df['readmitted'].value_counts()

C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\2720739403.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['readmitted'] = df['readmitted'].replace({ 'NO': 0, '>30': 0, '<30': 1}).astype(int)


readmitted
0    90409
1    11357
Name: count, dtype: int64

In [10]:
df.groupby('readmitted')[['number_inpatient', 'number_emergency', 'number_diagnoses', 'time_in_hospital']].mean()
# roughly inspecting if there is a connecting with readmission for these feature by the difference in their means
# number_inpatient and number_emergency seem to have some level of distinction

,number_inpatient,number_emergency,number_diagnoses,time_in_hospital
readmitted,,,,
0,0.561648,0.177803,7.388667,4.349224
1,1.224003,0.357313,7.692789,4.768249


In [11]:
print(df['encounter_id'].nunique())
print(df['patient_nbr'].nunique())
print(len(df))

101766
71518
101766


In [12]:
df.select_dtypes(include='object').columns          # check which columns need feature encoding

Index(['race', 'gender', 'age', 'payer_code', 'diag_1', 'diag_2', 'diag_3',
       'A1Cresult', 'metformin', 'glipizide', 'glyburide', 'insulin', 'change',
       'diabetesMed'],
      dtype='object')

In [13]:
df = df.drop(['race', 'gender', 'payer_code'], axis=1)
# these cols get dropped after inspection from Excel notebook for no noticeable variation between different values

In [14]:
# A1Cresult gets converted to Boolean since there are around 80k missing values out of 100k
df['A1Cresult'] = df['A1Cresult'].replace(to_replace=['>7', '>8', 'Norm'], value=1)
df['A1Cresult'] = df['A1Cresult'].fillna(0)

C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\1498275357.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['A1Cresult'] = df['A1Cresult'].replace(to_replace=['>7', '>8', 'Norm'], value=1)


In [15]:
# encoding age 
age_originals = ['[90-100)', '[80-90)', '[70-80)', '[60-70)', '[50-60)', '[40-50)', '[30-40)', '[20-30)', '[10-20)', '[0-10)']
age_encoded = [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
df['age'] = df['age'].replace(age_originals, age_encoded).astype(int)

C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\283404886.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['age'] = df['age'].replace(age_originals, age_encoded).astype(int)


In [16]:
# double checking the values of each col to see if encoding can be done simultaneously
print(df['change'].value_counts())
print(df['diabetesMed'].value_counts())
print(df['metformin'].value_counts())
print(df['glipizide'].value_counts())
print(df['glyburide'].value_counts())
print(df['insulin'].value_counts())

change
No    54755
Ch    47011
Name: count, dtype: int64
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64
metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: count, dtype: int64
glipizide
No        89080
Steady    11356
Up          770
Down        560
Name: count, dtype: int64
glyburide
No        91116
Steady     9274
Up          812
Down        564
Name: count, dtype: int64
insulin
No        47383
Steady    30849
Down      12218
Up        11316
Name: count, dtype: int64


In [17]:
# encoding change and diabetesMed separately
df['change'] = df['change'].replace(['Ch', 'No'], [1, 0]).astype(int)
df['diabetesMed'] = df['diabetesMed'].replace(['Yes', 'No'], [1, 0]).astype(int)

C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\2988841724.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['change'] = df['change'].replace(['Ch', 'No'], [1, 0]).astype(int)
C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\2988841724.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['diabetesMed'] = df['diabetesMed'].replace(['Yes', 'No'], [1, 0]).astype(int)


In [18]:
# simultaneously encoding cols with shared values
df[['metformin', 'glipizide', 'glyburide', 'insulin']] = df[['metformin', 'glipizide', 'glyburide', 'insulin']].replace(['No', 'Steady', 'Down', 'Up'], [0, 1, 2, 3]).astype(int)

C:\Users\Khin Lay Kywe\AppData\Local\Temp\ipykernel_17224\1910696515.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[['metformin', 'glipizide', 'glyburide', 'insulin']] = df[['metformin', 'glipizide', 'glyburide', 'insulin']].replace(['No', 'Steady', 'Down', 'Up'], [0, 1, 2, 3]).astype(int)


In [19]:
# performing one-hot encoding on the diagnoses cols
# one-hot encoding was chosen to prevent the model from thinking the order/value of the col had significance over the other
# e.g. preventing "values of 5 or higher are more likely to be readmitted" when the numbers do not correlate to the original diagnoses at all
df = pd.get_dummies(df, columns=['diag_1', 'diag_2', 'diag_3'], dtype=int)

In [20]:
# checking that the whole df has been encoded correctly
df.select_dtypes(include='object').columns

Index([], dtype='object')

In [21]:
# doing a train-test split by first splitting patient_id
# so that the model does not see the same patient from the train_df and test_df
from sklearn.model_selection import train_test_split
unique_patients = df['patient_nbr'].unique()
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2)
train_mask = df['patient_nbr'].isin(train_patients)                             # 1D array of T/F values
train_df = df[train_mask]                                                       # based on 1D array, we keep or discard so visits for patients all go into train or test without being split (data leakage)

test_mask = df['patient_nbr'].isin(test_patients)
test_df = df[test_mask]

x_train = train_df.drop('readmitted', axis=1)
y_train = train_df['readmitted']

x_test = test_df.drop('readmitted', axis=1)
y_test = test_df['readmitted']

In [22]:
# confirm there are no patients appearing simultaneously in both sets
print(train_df.shape, test_df.shape)
print(set(train_df['patient_nbr']) & set(test_df['patient_nbr']))

(81386, 79) (20380, 79)
set()


In [23]:
# patient_nbr and encounter_id can be dropped since they do not hold any correlation to our target variable
# i.e. we don't want the model memorizing patterns like 'patient_id between 10000 and 15000 always get readmitted'
x_train = x_train.drop(['patient_nbr', 'encounter_id'], axis=1)
x_test = x_test.drop(['patient_nbr', 'encounter_id'], axis=1)

In [24]:
# training a base model to benchmark against
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)

from sklearn.metrics import classification_report
y_pred = rf.predict(x_test)
print(classification_report(y_test, y_pred))

# due to the class imbalance in the data for 0:1, the accuracy is at 89% even for the baseline model
# precision is true positive hit rate out of all true and false positives predicted
# recall is true positive prediction rate out of all the actual positives in the sample

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18080
           1       0.64      0.00      0.01      2300

    accuracy                           0.89     20380
   macro avg       0.76      0.50      0.47     20380
weighted avg       0.86      0.89      0.83     20380



In [25]:
# using a balanced weight to overrepresent the undersamepled categories
rf_balanced = RandomForestClassifier(n_estimators=20, random_state=42, class_weight='balanced')
rf_balanced.fit(x_train, y_train)
from sklearn.metrics import classification_report
y_pred = rf_balanced.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18080
           1       0.48      0.00      0.01      2300

    accuracy                           0.89     20380
   macro avg       0.68      0.50      0.47     20380
weighted avg       0.84      0.89      0.84     20380



In [26]:
# using SMOTE to artifically create more data points for the undersampled target value (readmitted within <30 days)
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
x_train_sm, y_train_sm = sm.fit_resample(x_train, y_train)
y_train_sm.value_counts()

readmitted
0    72329
1    72329
Name: count, dtype: int64

In [27]:
rf_smote = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_smote.fit(x_train_sm, y_train_sm)
y_proba_sm = rf_smote.predict_proba(x_test)[:, 1]
print(y_proba_sm.max())             # the highest probability of a data point predicted to be 1
print(y_proba_sm.mean())            # the average probability of a data point predicted to be 1
pd.Series(y_proba_sm).describe()    # see full distribution to decide classification boundaries

0.9
0.16475275596990513


count    20380.000000
mean         0.164753
std          0.089403
min          0.000000
25%          0.100000
50%          0.150000
75%          0.220000
max          0.900000
dtype: float64

In [28]:
# precision is true positive hit rate out of all true and false positives predicted
# recall is true positive prediction rate out of all the actual positives in the sample
# testing thresholds to find the best f1 value since 0.5 wouldn't be a good predictor since the mean is 0.16

for threshold in [0.3, 0.25, 0.2, 0.15, 0.1]:
    y_pred_adjusted = (y_proba_sm > threshold).astype(int)
    report = classification_report(y_test, y_pred_adjusted, output_dict=True)
    print(f"Threshold: {threshold:.2f} | Recall: {report['1']['recall']:.2f} | Precision: {report['1']['precision']:.2f} | F1: {report['1']['f1-score']:.2f}")

Threshold: 0.30 | Recall: 0.13 | Precision: 0.21 | F1: 0.16
Threshold: 0.25 | Recall: 0.23 | Precision: 0.18 | F1: 0.20
Threshold: 0.20 | Recall: 0.40 | Precision: 0.16 | F1: 0.23
Threshold: 0.15 | Recall: 0.62 | Precision: 0.14 | F1: 0.23
Threshold: 0.10 | Recall: 0.83 | Precision: 0.13 | F1: 0.22


In [29]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score(y_test, y_proba_sm))
# this means my model is able to identify the readmitted patient 61% of the time

0.6047275875336668


In [30]:
# storing away the model and the feature columns
import joblib, os
os.makedirs('models', exist_ok=True)
joblib.dump(rf_smote, 'models/rf_smote.pkl')
joblib.dump(x_train.columns.tolist(), 'models/feature_columns.pkl')

['models/feature_columns.pkl']

In [31]:
# gets the 20 highest weighted features from the model to see what to put in the user form
importances = pd.Series(rf_smote.feature_importances_, index=x_train.columns)
importances.sort_values(ascending=False).head(20)

num_lab_procedures                                                              0.049803
A1Cresult                                                                       0.046412
num_medications                                                                 0.045124
diag_3_Endocrine, nutritional and metabolic diseases, and immunity disorders    0.040819
diag_2_Endocrine, nutritional and metabolic diseases, and immunity disorders    0.036003
diag_3_Diseases of the circulatory system                                       0.035084
time_in_hospital                                                                0.032350
diag_2_Diseases of the circulatory system                                       0.030725
age                                                                             0.029901
discharge_disposition_id                                                        0.025234
metformin                                                                       0.024682
number_inpatient     

In [32]:
# calculating mode to be set as default for the cols that will not be filled by user on the dashboard
non_input_cols = [col for col in x_train.columns if col not in [
    'age', 'num_lab_procedures', 'num_medications', 
    'number_diagnoses', 'metformin', 'insulin', 'number_inpatient',
    'A1Cresult'
] and not col.startswith('diag_')]

defaults = x_train[non_input_cols].mode().iloc[0].to_dict()
print(defaults)

{'admission_type_id': 1, 'discharge_disposition_id': 1, 'admission_source_id': 7, 'time_in_hospital': 3, 'num_procedures': 0, 'number_outpatient': 0, 'number_emergency': 0, 'glipizide': 0, 'glyburide': 0, 'change': 0, 'diabetesMed': 1}


In [33]:
# saving the default values calculated above
joblib.dump(defaults, 'models/defaults.pkl')

['models/defaults.pkl']